In [0]:
dbutils.library.restartPython()

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.scdfunctions import scd_type_2

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("bronze_to_silver_facts")

logger.info("Starting Bronze to Silver Fact processing")

# ============================================================
# 2. READ CONFIG
# ============================================================
config_path = "/Workspace/Users/shreyassrikanth997@gmail.com/Employee-Data-Platform/src/silver/config/config_facts.json"

with open(config_path, "r") as file:
    configs = json.load(file)

# ============================================================
# 3. DEDUPLICATION FUNCTION
# ============================================================
def apply_deduplication(df, dedup_config):

    method = dedup_config["method"]

    # ---------- KEEP LATEST RECORD PER KEY + BATCH ----------
    if method == "latest_record":

        window_spec = (
            Window
            .partitionBy(*dedup_config["partition_columns"])
            .orderBy(F.col(dedup_config["order_column"]).desc())
        )

        df = (
            df.withColumn("row_num", F.row_number().over(window_spec))
              .filter(F.col("row_num") == 1)
              .drop("row_num")
        )

    # ---------- DROP EXACT DUPLICATES ----------
    elif method == "drop_duplicates":

        df = df.dropDuplicates(dedup_config["columns"])

    else:
        raise ValueError(f"Unknown deduplication method: {method}")

    return df

# ============================================================
# 4. ADDITIONAL TRANSFORMATION FUNCTION
# ============================================================
def apply_additional_transformation(df, transformation):

    # ---------- SHRINKAGE DURATION IN MINUTES ----------
    if transformation == "calculate_shrinkage_duration":

        df = df.withColumn(
            "shrinkage_duration_minutes",
            (
                F.col("end_datetime").cast("long")
                - F.col("start_datetime").cast("long")
            ) / 60
        )

    return df

# ============================================================
# 5. MAIN BRONZE TO SILVER FUNCTION
# ============================================================
def process_fact(fact_name):

    logger.info(f"Starting processing for {fact_name}")

    # ---------- CONFIG SECTIONS ----------
    fact_config = configs[fact_name]

    source_schema = fact_config["source_schema"]
    table_prefix  = fact_config["source_table_prefix"]
    target_table  = fact_config["target_table"]
    key_columns   = fact_config["key_columns"]

    # ---------- FIND ALL BRONZE TABLES FOR THIS FACT ----------
    source_tables = sorted(
        f"{source_schema}.{row.tableName}"
        for row in spark.sql(f"SHOW TABLES IN {source_schema} LIKE '{table_prefix}*'").collect()
    )

    if not source_tables:
        raise ValueError(f"No Bronze tables found for {fact_name} with prefix {table_prefix}")

    logger.info(f"Found {len(source_tables)} Bronze tables for {fact_name}: {source_tables}")

    # ---------- READ AND COMBINE BRONZE TABLES ----------
    df = spark.table(source_tables[0])

    for table_name in source_tables[1:]:
        df = df.unionByName(spark.table(table_name))

    logger.info(f"Combined Bronze tables for {fact_name}")

    # ---------- DEDUPLICATION (within each batch only) ----------
    df = apply_deduplication(
        df,
        fact_config["deduplication"]
    )

    logger.info(f"Deduplication completed for {fact_name}")

    # ---------- DROP TECHNICAL COLUMNS ----------
    df = df.drop(*fact_config["drop_columns"])

    # ---------- ADDITIONAL TRANSFORMATION ----------
    df = apply_additional_transformation(
        df,
        fact_config["additional_transformation"]
    )

    logger.info(f"Transformations completed for {fact_name}")

    # ---------- CREATE SILVER SCHEMA IF DOES NOT EXIST ----------
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS employeedataplatform.silver
    """)

    # ---------- APPLY SCD TYPE 2 ----------
    scd_type_2(
        spark,
        df,
        target_table,
        key_columns
    )

    logger.info(f"Completed processing for {fact_name}")

    return spark.table(target_table)

# ============================================================
# 6. RUN FACTWORK
# ============================================================
fact_work = process_fact("fact_work")
print("Total Silver records:", fact_work.count())

# ============================================================
# 7. RUN FACTSHRINKAGE
# ============================================================
fact_shrinkage = process_fact("fact_shrinkage")
print("Total Silver records:", fact_shrinkage.count())

# ============================================================
# 8. RUN FACTSHIFT
# ============================================================
fact_shift = process_fact("fact_shift")
print("Total Silver records:", fact_shift.count())

# ============================================================
# 9. COMPLETED
# ============================================================
logger.info("Bronze to Silver Fact processing completed")